In [ ]:
import random
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, spearmanr

In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/all-MiniLM-L6-v2"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 128 if device == "mps" else 64
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})

In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
df["sentence1"] = df["sentence1"].astype(str).str.strip()
df["sentence2"] = df["sentence2"].astype(str).str.strip()
df["label"] = df["label"].astype(np.float32)

original_count = int(len(df))

canonical_left = np.where(df["sentence1"].to_numpy() <= df["sentence2"].to_numpy(), df["sentence1"].to_numpy(), df["sentence2"].to_numpy())
canonical_right = np.where(df["sentence1"].to_numpy() <= df["sentence2"].to_numpy(), df["sentence2"].to_numpy(), df["sentence1"].to_numpy())
df["pair_key_left"] = canonical_left
df["pair_key_right"] = canonical_right

dedup_df = (
    df.drop_duplicates(subset=["pair_key_left", "pair_key_right"], keep="first")
    .reset_index(drop=True)
    .copy()
)

dedup_count = int(len(dedup_df))
duplicates_removed = int(original_count - dedup_count)
coverage_fraction = float(dedup_count / original_count) if original_count > 0 else 0.0

print({
    "original_examples": original_count,
    "deduplicated_examples": dedup_count,
    "duplicates_removed": duplicates_removed,
    "coverage_fraction": round(coverage_fraction, 6),
    "columns": dedup_df.columns.tolist(),
})
print(dedup_df[["sentence1", "sentence2", "label"]].head())

In [ ]:
duplicate_impact = (
    df.groupby(["pair_key_left", "pair_key_right"], observed=False)
    .agg(
        occurrences=("label", "size"),
        label_mean=("label", "mean"),
        label_std=("label", "std"),
        first_sentence1=("sentence1", "first"),
        first_sentence2=("sentence2", "first"),
    )
    .reset_index(drop=True)
)

duplicate_groups = duplicate_impact[duplicate_impact["occurrences"] > 1].copy()
duplicate_groups["label_std"] = duplicate_groups["label_std"].fillna(0.0)

print({
    "duplicate_pair_groups": int(len(duplicate_groups)),
    "duplicate_rows_removed": duplicates_removed,
    "max_duplicate_group_size": int(duplicate_groups["occurrences"].max()) if len(duplicate_groups) else 1,
    "mean_duplicate_group_size": round(float(duplicate_groups["occurrences"].mean()), 4) if len(duplicate_groups) else 1.0,
    "mean_label_std_within_duplicate_groups": round(float(duplicate_groups["label_std"].mean()), 6) if len(duplicate_groups) else 0.0,
})

if len(duplicate_groups):
    print(
        duplicate_groups[["first_sentence1", "first_sentence2", "occurrences", "label_mean", "label_std"]]
        .sort_values(by=["occurrences", "label_std"], ascending=[False, False])
        .head(10)
        .reset_index(drop=True)
    )

In [ ]:
model = SentenceTransformer(model_name, device=device)
model.eval()
print(model_name)

In [ ]:
eval_df = dedup_df[["sentence1", "sentence2", "label"]].copy()
sentences1 = eval_df["sentence1"].tolist()
sentences2 = eval_df["sentence2"].tolist()
labels = eval_df["label"].to_numpy(dtype=np.float32)

emb1 = model.encode(
    sentences1,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

emb2 = model.encode(
    sentences2,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

cosine_similarity = np.sum(emb1 * emb2, axis=1).astype(np.float32)
predicted_score_0_5 = (2.5 * (cosine_similarity + 1.0)).astype(np.float32)
absolute_error = np.abs(predicted_score_0_5 - labels).astype(np.float32)
squared_error = np.square(predicted_score_0_5 - labels).astype(np.float32)

results_df = eval_df.copy()
results_df["cosine_similarity"] = cosine_similarity
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["absolute_error"] = absolute_error
results_df["squared_error"] = squared_error
results_df["signed_error"] = results_df["predicted_score_0_5"] - results_df["label"]
results_df["agreement_gap"] = np.abs(results_df["cosine_similarity"] - (results_df["label"] / 2.5 - 1.0)).astype(np.float32)

print(results_df[["sentence1", "sentence2", "label", "cosine_similarity", "predicted_score_0_5", "absolute_error"]].head(10))

In [ ]:
pearson_corr = pearsonr(results_df["predicted_score_0_5"], results_df["label"]).statistic
spearman_corr = spearmanr(results_df["predicted_score_0_5"], results_df["label"]).statistic
mae = float(results_df["absolute_error"].mean())
rmse = float(np.sqrt(results_df["squared_error"].mean()))

label_bin_edges = [-0.001, 1.0, 2.0, 3.0, 4.0, 5.001]
label_bin_names = ["[0,1)", "[1,2)", "[2,3)", "[3,4)", "[4,5]"]
results_df["label_bin"] = pd.cut(
    results_df["label"],
    bins=label_bin_edges,
    labels=label_bin_names,
    include_lowest=True,
    right=False,
)

bin_agg = (
    results_df.groupby("label_bin", observed=False)
    .agg(
        count=("label", "size"),
        label_mean=("label", "mean"),
        pred_mean=("predicted_score_0_5", "mean"),
        cosine_mean=("cosine_similarity", "mean"),
        mae=("absolute_error", "mean"),
        rmse=("squared_error", lambda x: float(np.sqrt(np.mean(x)))),
        signed_error_mean=("signed_error", "mean"),
    )
    .reset_index()
)

coverage_report = {
    "original_examples": original_count,
    "evaluated_examples_after_dedup": dedup_count,
    "duplicates_removed": duplicates_removed,
    "coverage_fraction": round(coverage_fraction, 6),
}

print(coverage_report)
print({
    "pearson_correlation": round(float(pearson_corr), 6),
    "spearman_correlation": round(float(spearman_corr), 6),
    "mae_0_5": round(mae, 6),
    "rmse_0_5": round(rmse, 6),
})
print(bin_agg)

In [ ]:
best_examples = (
    results_df.sort_values(
        by=["absolute_error", "agreement_gap", "label"],
        ascending=[True, True, False],
    )
    [["sentence1", "sentence2", "label", "predicted_score_0_5", "cosine_similarity", "absolute_error", "signed_error"]]
    .head(10)
    .reset_index(drop=True)
)

worst_examples = (
    results_df.sort_values(
        by=["absolute_error", "agreement_gap", "label"],
        ascending=[False, False, False],
    )
    [["sentence1", "sentence2", "label", "predicted_score_0_5", "cosine_similarity", "absolute_error", "signed_error"]]
    .head(10)
    .reset_index(drop=True)
)

pd.set_option("display.max_colwidth", 160)
print("BEST_AGREEMENT_EXAMPLES")
print(best_examples)
print("\nWORST_AGREEMENT_EXAMPLES")
print(worst_examples)

In [ ]:
runtime_seconds = time.time() - start_time

summary = {
    "device_used": device,
    "model_name": model_name,
    "dataset_split": f"{dataset_name}/{dataset_config}/{split_name}",
    "original_examples": int(original_count),
    "evaluated_examples_after_dedup": int(dedup_count),
    "duplicates_removed": int(duplicates_removed),
    "coverage_fraction": round(float(coverage_fraction), 6),
    "pearson_correlation": round(float(pearson_corr), 6),
    "spearman_correlation": round(float(spearman_corr), 6),
    "mae_0_5": round(mae, 6),
    "rmse_0_5": round(rmse, 6),
    "runtime_seconds": round(float(runtime_seconds), 2),
}

print(summary)